# The silent 200 — a quality signal on the scrape pipeline's own output

A `200 OK` doesn't mean the content is good. A scrape can succeed — `success: true`, `statusCode: 200` — and still hand back an empty shell, an unhydrated JS skeleton, mojibake, an anti-bot interstitial, or pure boilerplate. That's a **silent 200**.

`quality(doc)` is a cheap, deterministic read on exactly that. This notebook imports the **real shipping function** — [`apps/api/src/lib/quality.ts`](../../apps/api/src/lib/quality.ts), pure and dependency-free — and runs it live.

> Launch `deno jupyter` (or `jupyter lab` with the Deno kernel) **from this directory** so the relative import resolves.

In [1]:
import { quality } from "../../apps/api/src/lib/quality.ts";
// the exact function that ships in the scrape pipeline — pure, no imports, no network
const r = quality({
  markdown: "# Firecrawl\n\nFirecrawl turns entire websites into clean, LLM-ready markdown. " +
    "Point it at a URL and get back structured content you can feed straight to a model.",
  html: "<h1>Firecrawl</h1><p>…</p>",
});
console.log("rating:", r.rating, "| score:", r.score, "| findings:", r.findings.map((f) => f.code));

rating: A | score: 100 | findings: []


## The five silent-200 shapes

Each heuristic targets a recurring, documented customer issue. A finding is a *symptom* plus evidence, a hint, and (where useful) a suggested fix — and it cites the issues it addresses. One high-severity finding caps the grade at `F` (a fatal flaw isn't averaged away).

In [2]:
import { quality, type QualityInput } from "../../apps/api/src/lib/quality.ts";

// authentic mojibake: UTF-8 bytes decoded as Latin-1 (Deno-native, no Node Buffer)
const utf8 = new TextEncoder().encode("こんにちは、世界。製品の説明です。Größe café déjà");
const mojibake = new TextDecoder("latin1").decode(utf8).repeat(4);

const cases: Record<string, QualityInput> = {
  EMPTY: { markdown: "", html: "<body></body>", rawHtml: "<body></body>" },
  JS_SHELL: {
    markdown: "Loading…",
    html: "<div id=root></div>",
    rawHtml: "<!doctype html>" + "<script src=/a.js></script>".repeat(12) +
      "<div id=root></div>" + "x".repeat(40000),
  },
  GARBLED: { markdown: mojibake },
  SOFT_BOT_WALL: {
    markdown: "Checking your browser before accessing the site. Please enable JavaScript and cookies to continue.",
    html: "<title>Just a moment…</title>",
  },
  HIGH_BOILERPLATE: {
    markdown: [
      "[Home](/) [Products](/p) [Pricing](/pr) [About](/a) [Login](/l) [Sign up](/s)",
      "We use cookies. Accept all · Privacy Policy · Cookie Policy",
      "Subscribe to our newsletter. Follow us.",
      "© 2026 Acme Inc. All rights reserved. Terms · Sitemap · Careers",
      "Widget.",
    ].join("\n\n"),
  },
};

console.log("shape              rating  finding             cites issues");
console.log("─".repeat(62));
for (const [name, input] of Object.entries(cases)) {
  const r = quality(input);
  const f = r.findings[0];
  console.log(
    name.padEnd(18),
    (r.rating + " ").padEnd(7),
    (f?.code ?? "—").padEnd(19),
    f?.issues.map((n) => "#" + n).join(" ") ?? "",
  );
}

shape              rating  finding             cites issues
──────────────────────────────────────────────────────────────
EMPTY              F       EMPTY               #385 #684 #666 #1297
JS_SHELL           F       JS_SHELL            #1345 #1297 #2375
GARBLED            F       GARBLED             #1142 #1277 #547
SOFT_BOT_WALL      F       SOFT_BOT_WALL       #2350 #495 #2413
HIGH_BOILERPLATE   B       HIGH_BOILERPLATE    #284 #288 #1564 #540


## What makes it trustworthy — the false-positive guards

A naive matcher would flag real i18n text as mojibake, or an article *about* Cloudflare as a bot wall. These near-misses are the whole game — the signal is only useful if it doesn't cry wolf. Two of these guards (the 404-image case and the long-article case) were **false positives the live corpus run caught**, then fixed.

In [3]:
import { quality, type QualityInput } from "../../apps/api/src/lib/quality.ts";

// near-miss content a naive matcher would false-positive on — these are what make the signal trustworthy
const guards: Record<string, QualityInput> = {
  "real fr/de i18n (accents)": {
    markdown: "La société française a développé une méthode très élégante pour résumer des " +
      "données. Café, thé, crème brûlée — et la Größe du résultat était excellente.",
  },
  "article ABOUT cloudflare": {
    markdown: ("Cloudflare is a CDN and security provider. Many sites put it in front of their " +
      "origin to mitigate DDoS attacks, and some present a CAPTCHA challenge to suspected bots. ").repeat(2),
    html: "<article></article>",
  },
  "404 linking a cf image": {
    markdown: "Page not found\n\nError Code: 404\n\n![](https://blog.cloudflare.com/images/404.svg)",
  },
  "long article quoting wall copy": {
    markdown: ("Cloudflare provides DDoS protection. The block page is titled 'Attention Required! " +
      "| Cloudflare' and shows a Ray ID. The classic interstitial reads 'Checking your browser " +
      "before accessing the site.' Researchers quote this verbatim when explaining how anti-bot systems decide whether to verify you are human. ").repeat(8),
  },
};

console.log("guard                             rating  result");
console.log("─".repeat(58));
for (const [name, input] of Object.entries(guards)) {
  const r = quality(input);
  console.log(
    name.padEnd(33),
    (r.rating + " ").padEnd(7),
    r.findings.length ? "FLAGGED " + r.findings.map((f) => f.code).join(",") : "clean ✓",
  );
}

guard                             rating  result
──────────────────────────────────────────────────────────
real fr/de i18n (accents)         A       clean ✓
article ABOUT cloudflare          A       clean ✓
404 linking a cf image            A       clean ✓
long article quoting wall copy    A       clean ✓


## The live engine delta

Forcing `fetch` vs `playwright` on the same URL and comparing `quality()` is exactly the input an engine router — or the offline engpicker calibration — would route on. Below, read from the committed corpus run (`out/results.json`): where plain fetch silently returned a near-empty 200, and a browser recovered the content.

In [4]:
// the live engine delta, read from the committed corpus run (out/results.json)
type R = { url: string; engine: string; rating: string | null; findings: string[]; mdChars: number; error: string | null };
let results: R[] = [];
try {
  results = JSON.parse(await Deno.readTextFile("out/results.json"));
} catch {
  console.log("run `npm run corpus:engines` first to populate out/results.json");
}

if (results.length) {
  const at = (u: string, e: string) => results.find((r) => r.url === u && r.engine === e);
  const urls = [...new Set(results.map((r) => r.url))];
  const flips: string[] = [], recovery: { line: string; ratio: number }[] = [];
  for (const u of urls) {
    const fe = at(u, "fetch"), pw = at(u, "playwright");
    if (!fe || !pw || fe.error || pw.error) continue;
    const short = u.replace(/^https?:\/\//, "");
    if (fe.rating === "F" && (pw.rating === "A" || pw.rating === "B")) flips.push(short);
    const ratio = pw.mdChars / Math.max(fe.mdChars, 1);
    if (ratio >= 2 && pw.mdChars - fe.mdChars >= 200) {
      recovery.push({ line: `${short.padEnd(40)} ${fe.mdChars}c → ${pw.mdChars}c  (${ratio.toFixed(1)}×)`, ratio });
    }
  }
  const dist: Record<string, number> = {};
  for (const r of results) if (r.rating) dist[r.rating] = (dist[r.rating] ?? 0) + 1;

  console.log("grade distribution across", results.length, "engine-calls:", JSON.stringify(dist));
  console.log("\ngrade flips  (fetch silently 200s → browser recovers it):");
  for (const u of flips) console.log("  •", u);
  console.log("\ncontent recovery  (browser ≥2× the text fetch saw):");
  for (const r of recovery.sort((a, b) => b.ratio - a.ratio)) console.log("  •", r.line);
}

grade distribution across 72 engine-calls: {"F":4,"A":59,"B":2}

grade flips  (fetch silently 200s → browser recovers it):
  • excalidraw.com
  • vscode.dev

content recovery  (browser ≥2× the text fetch saw):
  • vscode.dev                               0c → 896c  (896.0×)
  • excalidraw.com                           21c → 651c  (31.0×)
  • www.desmos.com/calculator                39c → 469c  (12.0×)
  • quotes.toscrape.com/js/                  143c → 1574c  (11.0×)
  • www.lsu.edu                              4287c → 8861c  (2.1×)


## Where it goes next

`quality()` is a primitive, not a product. Today it surfaces as a user-facing `quality` field, off by default behind `__experimental_quality`. The same function can feed **engine selection**: route on `quality(fetch)` vs `quality(playwright)` instead of a bare similarity score, and gate retries on the symptom. The deterministic pass/fail gate is [`quality.test.ts`](../../apps/api/src/lib/quality.test.ts); this notebook is the guided tour.